# OpenRouter Usage Cost Tracker

**Date:** 2026-02-08  
**Purpose:** Track OpenRouter API spending per forecast session
**Claude Code Session and Description**: in C:\Users\Donni\projects\metac_bot_Spring_2026\conversation and context docs

This notebook:
1. Queries the OpenRouter API for current credit balance (`limit_remaining`)
2. Reads prior history from a local Excel file
3. Prompts for number of questions forecast this session
4. Calculates usage, cost per question, and cumulative stats
5. Reports on screen and saves new row to Excel history

---

## Setup & Configuration

In [1]:
# Install openpyxl if needed (uncomment to install)
# !pip install openpyxl

In [2]:
import os
import urllib.request
import json
from openpyxl import load_workbook, Workbook
from datetime import datetime
from pathlib import Path

In [3]:
# Configuration
HISTORY_FILE = Path('openrouter_cost_history.xlsx')  # Save in current directory
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')  # Load from environment

# If API key not in environment, set it here (not recommended for version control)
# OPENROUTER_API_KEY = 'sk-or-v1-your-key-here'

if not OPENROUTER_API_KEY:
    print("⚠️  WARNING: OPENROUTER_API_KEY not set!")
    print("Set it in your environment or directly in the cell above.")
else:
    print("✅ API key loaded")

✅ API key loaded


## Get Current Balance from OpenRouter API

In [4]:
def get_openrouter_balance(api_key):
    """Fetch current limit_remaining from OpenRouter API."""
    url = "https://openrouter.ai/api/v1/auth/key"
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {api_key}"})
    
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read())
        return data['data']['limit_remaining']
    except Exception as e:
        print(f"❌ Error fetching balance: {e}")
        return None

current_balance = get_openrouter_balance(OPENROUTER_API_KEY)

if current_balance is not None:
    print(f"💰 Current balance (limit_remaining): ${current_balance:.4f}")
else:
    print("Failed to fetch balance. Check API key and connection.")

💰 Current balance (limit_remaining): $544.2246


## Load History from Excel

In [5]:
# Excel column schema
COLUMNS = [
    'datetime', 'prior_balance', 'current_balance',
    'cost_this_session', 'questions_this_session', 'cost_per_question_session',
    'total_questions_inception', 'avg_cost_per_question_inception',
    'total_cost_inception'
]

def load_history():
    """Load or create the Excel history workbook."""
    if HISTORY_FILE.exists():
        wb = load_workbook(HISTORY_FILE)
        ws = wb.active
        rows = list(ws.iter_rows(min_row=2, values_only=True))  # Skip header
        return wb, ws, rows
    else:
        wb = Workbook()
        ws = wb.active
        ws.append(COLUMNS)  # Create header row
        return wb, ws, []

wb, ws, history = load_history()

print("\n" + "="*60)
print("HISTORY LOADED")
print("="*60)

if history:
    last_row = history[-1]
    prior_balance = last_row[2]        # current_balance from last session = prior for this one
    total_questions_prior = last_row[6] or 0  # total_questions_inception
    total_cost_prior = last_row[8] or 0.0     # total_cost_inception
    
    print(f"Prior balance:         ${prior_balance:.4f}")
    print(f"Prior total questions: {total_questions_prior}")
    print(f"Prior total cost:      ${total_cost_prior:.4f}")
    print(f"Total sessions logged: {len(history)}")
else:
    prior_balance = current_balance if current_balance else 0.0
    total_questions_prior = 0
    total_cost_prior = 0.0
    print("📝 No prior history - starting fresh.")
    print(f"Initial balance:       ${prior_balance:.4f}")

print("="*60)


HISTORY LOADED
📝 No prior history - starting fresh.
Initial balance:       $544.2246


## Calculate Cost and Report

**VERIFY THE FOLLOWING CELL SHOULD BE REMOVED OR UPDATED AFTER THE FIRST RUN**

In [6]:
# preliminary start balance as of 01/23/2026
prior_balance = 581.77

In [7]:
# Quick estimate of number of questions so far
spring = 68
mini = 34
practice = 15
question_count = spring + mini + practice
question_count

117

In [8]:
# Prompt for question count
questions_this_session = int(input("How many questions were forecast this session? (0 if none): "))

# Calculate costs
cost_this_session = prior_balance - current_balance
cost_per_q_session = cost_this_session / questions_this_session if questions_this_session > 0 else 0.0
total_questions = total_questions_prior + questions_this_session
total_cost = total_cost_prior + cost_this_session
avg_cost_per_q = total_cost / total_questions if total_questions > 0 else 0.0

# Display report
print("\n" + "="*60)
print("📊 SESSION COST REPORT")
print("="*60)
print(f"Cost this session:             ${cost_this_session:.4f}")
print(f"Questions this session:        {questions_this_session}")
print(f"Avg cost/question (session):   ${cost_per_q_session:.4f}")
print("-"*60)
print(f"Total cost (inception):        ${total_cost:.4f}")
print(f"Total questions (inception):   {total_questions}")
print(f"Avg cost/question (total):     ${avg_cost_per_q:.4f}")
print("="*60)

How many questions were forecast this session? (0 if none):  117



📊 SESSION COST REPORT
Cost this session:             $37.5454
Questions this session:        117
Avg cost/question (session):   $0.3209
------------------------------------------------------------
Total cost (inception):        $37.5454
Total questions (inception):   117
Avg cost/question (total):     $0.3209


## Save to Excel

In [9]:
# Add new row with timestamp
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
ws.append([
    now, 
    prior_balance, 
    current_balance,
    cost_this_session, 
    questions_this_session, 
    cost_per_q_session,
    total_questions, 
    avg_cost_per_q, 
    total_cost
])

wb.save(HISTORY_FILE)
print(f"\n✅ History saved to: {HISTORY_FILE.absolute()}")


✅ History saved to: C:\Users\Donni\projects\metac_bot_Spring_2026\jupyter\openrouter_cost_history.xlsx


---

## Excel Schema Reference

| Column | Description |
|--------|-------------|
| `datetime` | Timestamp of this session |
| `prior_balance` | limit_remaining from last session |
| `current_balance` | limit_remaining now |
| `cost_this_session` | prior - current |
| `questions_this_session` | User-entered count |
| `cost_per_question_session` | cost / questions for this session |
| `total_questions_inception` | Cumulative question count |
| `avg_cost_per_question_inception` | total_cost / total_questions |
| `total_cost_inception` | Cumulative cost |